In [ ]:
!pip install PySastrawi

# Tugas STKI: Perbandingan TF-IDF dan VSM

Notebook ini membandingkan dua metode perankingan dokumen:
- **TF-IDF**: penjumlahan skor bobot term query pada setiap dokumen.
- **VSM**: cosine similarity antara vektor query dan vektor dokumen berbasis TF-IDF.

Semua komentar, label, dan output menggunakan Bahasa Indonesia.

## Install Library

Library inti sudah diinstal pada cell pertama menggunakan `PySastrawi`.

## Import & Konfigurasi

In [ ]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 140)

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Import dan konfigurasi selesai.')

## Dokumen Korpus

Korpus berisi 5 dokumen pendek Bahasa Indonesia dengan topik berbeda.

In [ ]:
dokumen = {
    'D1': 'Perusahaan teknologi mengembangkan kecerdasan buatan untuk analisis data pelanggan dan otomatisasi layanan digital di kota besar.',
    'D2': 'Tim sepak bola nasional menjalani latihan intensif untuk menghadapi turnamen regional dan meningkatkan ketahanan fisik pemain.',
    'D3': 'Sekolah menengah mulai menerapkan platform pembelajaran digital berbasis kecerdasan buatan untuk membantu evaluasi siswa.',
    'D4': 'Puskesmas mengadakan program edukasi kesehatan ibu dan anak serta kampanye gizi seimbang di wilayah pedesaan.',
    'D5': 'Komunitas lingkungan menanam mangrove di pesisir untuk mengurangi abrasi dan menjaga ekosistem laut tetap stabil.'
}

df_korpus = pd.DataFrame({
    'Dokumen': list(dokumen.keys()),
    'Teks Asli': list(dokumen.values())
})

display(df_korpus)

## Preprocessing

Pipeline `preprocess(text)` (wajib urut):
1. Case folding
2. Cleaning (regex: hapus angka dan tanda baca)
3. Tokenizing
4. Stopword Removal (`StopWordRemoverFactory`)
5. Stemming (`StemmerFactory`)

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    teks_tanpa_stopword = stopword_remover.remove(' '.join(tokens))
    tokens_tanpa_stopword = teks_tanpa_stopword.split()
    tokens_stem = [stemmer.stem(token) for token in tokens_tanpa_stopword]
    return tokens_stem

hasil_preproses = {nama_dok: preprocess(teks) for nama_dok, teks in dokumen.items()}

df_preproses = pd.DataFrame({
    'Dokumen': list(hasil_preproses.keys()),
    'Token Hasil Preprocessing': [' '.join(v) for v in hasil_preproses.values()]
})

display(df_preproses)

## Perhitungan TF-IDF Manual

Perhitungan dilakukan manual tanpa `sklearn/TfidfVectorizer`.

- TF(d, t) = frekuensi term `t` pada dokumen `d` / total token dokumen `d`
- IDF(t) = log10(N / df(t))
- TF-IDF(d, t) = TF(d, t) × IDF(t)

Output utama section ini adalah tabel lengkap **term × dokumen**.

In [ ]:
nama_dokumen = list(hasil_preproses.keys())
N = len(nama_dokumen)

vocab = sorted({term for tokens in hasil_preproses.values() for term in tokens})

# TF manual
tf = {}
for dok, tokens in hasil_preproses.items():
    counter = Counter(tokens)
    total_token = len(tokens) if len(tokens) > 0 else 1
    tf[dok] = {term: counter.get(term, 0) / total_token for term in vocab}

# DF manual
df = {term: sum(1 for dok in nama_dokumen if tf[dok][term] > 0) for term in vocab}

# IDF manual
idf = {term: math.log10(N / df[term]) if df[term] > 0 else 0.0 for term in vocab}

# TF-IDF manual
tfidf = {
    dok: {term: tf[dok][term] * idf[term] for term in vocab}
    for dok in nama_dokumen
}

# Tabel TF-IDF lengkap: term x dokumen
df_tfidf = pd.DataFrame(
    {dok: [tfidf[dok][term] for term in vocab] for dok in nama_dokumen},
    index=vocab
)
df_tfidf.index.name = 'Term'

print('Ukuran tabel TF-IDF (term x dokumen):', df_tfidf.shape)
display(df_tfidf)

## Ranking dengan TF-IDF

Skor dokumen dihitung dengan menjumlahkan bobot TF-IDF term-term query pada setiap dokumen.

In [ ]:
query = 'kecerdasan buatan pendidikan digital'
query_tokens = preprocess(query)
query_terms = [term for term in query_tokens if term in vocab]

skor_tfidf = {}
for dok in nama_dokumen:
    skor_tfidf[dok] = sum(tfidf[dok][term] for term in query_terms)

ranking_tfidf = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Skor TF-IDF': [skor_tfidf[d] for d in nama_dokumen]
}).sort_values('Skor TF-IDF', ascending=False).reset_index(drop=True)

ranking_tfidf['Ranking TF-IDF'] = ranking_tfidf.index + 1

print('Query:', query)
print('Term query setelah preprocessing:', query_terms)
display(ranking_tfidf)

## Ranking dengan VSM (Cosine Similarity)

Representasikan query dan dokumen sebagai vektor TF-IDF, kemudian hitung cosine similarity.

In [ ]:
def cosine_similarity_manual(vec_a, vec_b):
    pembilang = float(np.dot(vec_a, vec_b))
    penyebut = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    return pembilang / penyebut if penyebut != 0 else 0.0

counter_query = Counter(query_terms)
total_query = len(query_terms) if len(query_terms) > 0 else 1

tf_query = {term: counter_query.get(term, 0) / total_query for term in vocab}
vektor_query = np.array([tf_query[term] * idf[term] for term in vocab], dtype=float)

skor_vsm = {}
for dok in nama_dokumen:
    vektor_dok = np.array([tfidf[dok][term] for term in vocab], dtype=float)
    skor_vsm[dok] = cosine_similarity_manual(vektor_query, vektor_dok)

ranking_vsm = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Skor VSM': [skor_vsm[d] for d in nama_dokumen]
}).sort_values('Skor VSM', ascending=False).reset_index(drop=True)

ranking_vsm['Ranking VSM'] = ranking_vsm.index + 1
display(ranking_vsm)

## Perbandingan Hasil Ranking

Tabel berikut menampilkan perbandingan ranking TF-IDF dan VSM secara berdampingan.

In [ ]:
hasil_perbandingan = (
    ranking_tfidf[['Dokumen', 'Skor TF-IDF', 'Ranking TF-IDF']]
    .merge(ranking_vsm[['Dokumen', 'Skor VSM', 'Ranking VSM']], on='Dokumen', how='inner')
)

hasil_perbandingan = hasil_perbandingan.sort_values('Ranking TF-IDF').reset_index(drop=True)

display(hasil_perbandingan[['Dokumen', 'Skor TF-IDF', 'Ranking TF-IDF', 'Skor VSM', 'Ranking VSM']])

## Analisis & Kesimpulan

Berdasarkan tabel perbandingan, ranking TF-IDF dan VSM bisa berbeda pada dokumen tertentu.

- Metode **TF-IDF (penjumlahan skor)** cenderung menguntungkan dokumen yang memiliki banyak term query dengan bobot besar secara total.
- Metode **VSM (cosine similarity)** mempertimbangkan arah dan normalisasi panjang vektor, sehingga dokumen yang lebih fokus pada topik query bisa memiliki similarity lebih tinggi.
- Jika ada pergeseran posisi, dokumen yang bergeser menunjukkan perbedaan karakter kedua metode: akumulasi bobot vs kemiripan ter-normalisasi.